# Local Ollama workflow / 本機 Ollama 工作流程

This lesson checks, optionally runs, verifies, and inspects a three-day FE Walter diagnostic with `gemma3:4b`. It never runs a model when the notebook opens. / 本課程會檢查、選擇性執行、驗證並檢視三天的 FE Walter 診斷案例；開啟筆記本時不會自動執行模型。

The frozen inflow is simulated and already routed to the reservoir boundary. Initial storage (1992.61 acre-feet) and `UH=[1]` are experimental assumptions. Completion does not establish measured storage, calibrated routing, or policy acceptance. / 凍結入流為模擬資料，且已路由至水庫邊界。初始蓄水量（1992.61 acre-feet）與 `UH=[1]` 為實驗假設；完成執行不代表蓄水量實測、路由校準或政策獲得接受。

## Three methods / 三種方法

1. Deterministic benchmark: no model call; see `03_benchmark.ipynb`. / 決定性基準：不呼叫模型。
2. Zero-shot: original V3.7 system prompt and native response schema; see `04_zero_shot.ipynb`. / 零樣本：保留原始 V3.7 系統提示與回覆結構。
3. DKP: the same interface plus original conditional `reference_v3_7` retrieval; see `05_dkp.ipynb`. / DKP：相同介面，加上原始條件式 `reference_v3_7` 檢索。

Hydrology, shared physics, routing, and multi-reservoir extensions are covered in tutorials 01, 02, 06, and 07. / 水文、共用物理、路由與多水庫延伸請見教學 01、02、06、07。

In [ ]:
from pathlib import Path
import contextlib, io, json, sys

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'pyproject.toml').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the HydroOpsLM checkout. / 請從專案工作目錄開啟本筆記本。')
CONFIGS = {
    'zero_shot': ROOT / 'examples/fe_walter/zero_shot/experiment.ollama.yaml',
    'dkp': ROOT / 'examples/fe_walter/dkp/experiment.ollama.yaml',
}
OUTPUTS = {
    'zero_shot': ROOT / 'runs/hydroopslm-fe-walter-zero-shot-ollama-gemma3-4b-20261012',
    'dkp': ROOT / 'runs/hydroopslm-fe-walter-dkp-ollama-gemma3-4b-20261012',
}
METHOD = 'zero_shot'  # Change to 'dkp' for original conditional retrieval. / 改成 'dkp' 使用原始條件式檢索。
RUN_LIVE = False      # Explicit opt-in only. / 必須明確改成 True 才會呼叫本機 Ollama。

sys.path.insert(0, str(ROOT / 'src'))
from llm_reservoir_agent.workflow import main as experiment_cli

def cli(*args):
    captured = io.StringIO()
    with contextlib.redirect_stdout(captured):
        experiment_cli([*map(str, args)])
    return json.loads(captured.getvalue())

plan = cli('check', CONFIGS[METHOD])  # Offline validation only. / 僅離線驗證設定。
print(json.dumps(plan, indent=2, ensure_ascii=False))
assert plan['planned_llm_calls'] == plan['max_calls'] == 3
assert plan['start_date'] == '2016-01-01' and plan['end_date'] == '2016-01-03'


## Explicit run or verified reuse / 明確執行或驗證後重用

Keep `RUN_LIVE=False` to inspect an existing output without another request. Set it to `True` only after confirming Ollama is already running and `gemma3:4b` is installed. The CLI still requires `--allow-network`, permits at most three attempts, and never downloads a model. / 保持 `RUN_LIVE=False` 可重用既有輸出而不再發送請求。只有在確認 Ollama 已啟動且已安裝 `gemma3:4b` 後才改成 `True`。CLI 仍要求 `--allow-network`、最多三次嘗試，且不會下載模型。

In [ ]:
output = OUTPUTS[METHOD]
if output.exists():
    # Real local runs are verified without --require-offline. / 真實本機執行不加 --require-offline。
    verified = cli('verify', output)
    print('Verified existing output; no repeat request. / 已驗證既有輸出，未重複請求。')
elif not RUN_LIVE:
    raise RuntimeError(
        f'No saved output at {output}. Keep RUN_LIVE=False until Ollama and gemma3:4b are ready; '
        'then set RUN_LIVE=True for the explicit three-request diagnostic. / 找不到既有輸出；確認服務與模型後，明確改成 RUN_LIVE=True。')
else:
    if not plan['live_ready']:
        raise RuntimeError('Case is not scoped as ready: ' + json.dumps(plan['live_readiness'], ensure_ascii=False))
    result = cli('run', CONFIGS[METHOD], '--allow-network')
    verified = cli('verify', output)
    print('Completed and verified exactly one bounded diagnostic. / 已完成並驗證一次有界診斷。')
print(json.dumps(verified, indent=2, ensure_ascii=False))


## Complete saved decision flow / 完整保存的決策流程

The next cell prints the actual saved observation (`info`), system prompt, user prompt, native reply, requested action, current contract recheck, retained original audit, implemented physics, history, and next-day observation. Saved evidence and the current recheck are labeled separately. / 下一格完整列出實際保存的觀測、系統提示、使用者提示、原生回覆、請求動作、目前契約重檢、原始稽核、實作物理、歷史與下一日觀測；保存證據與目前重檢會分開標示。

In [ ]:
sys.path.insert(0, str(ROOT))
from tutorials.inspect_decision_flow import load_decision_flow

flow = load_decision_flow(output, reservoir_id='fe_walter', timestep=0)
for key in [
    'provider_evidence', 'info', 'system_prompt', 'prompt', 'response_schema',
    'saved_native_reply', 'saved_requested_action', 'current_recheck', 'saved_audit',
    'saved_implemented_physics', 'saved_native_history', 'next_info', 'trace_files',
]:
    print(f'\n=== {key} ===')
    print(json.dumps(flow[key], indent=2, ensure_ascii=False) if not isinstance(flow[key], str) else flow[key])
